## Feature Engineering

Import the required libraries

In [10]:
import pandas as pd
pd.set_option('display.max_columns', 100)


Import the cleaned dataset

In [11]:
df = pd.read_csv('flights_eda_df.csv')
df.sample(3)

,origin,City,destination,City_destination,distance_km,Name_airline,aircraft,query_date,departure_date,departure_clock_time,day_of_week_departure,month_departure,arrival_date,arrival_clock_time,days_until_departure,trip_duration_minutes,number_of_stops,base_price,departure_hour,arrival_hour,departure_time_period,arrival_time_period,duration_group
29800,YVR,Vancouver,YYZ,Toronto,3345.516314,WestJet,7M8,2026-03-08,2026-05-03,07:00:00,Sunday,5,2026-05-03,18:52:00,56,322,1,105,7,18,Early Morning,Evening,"[300, 360)"
10513,YOW,Ottawa,YVR,Vancouver,3552.042539,Air Canada,E75,2026-03-08,2026-06-29,14:25:00,Monday,6,2026-06-29,20:20:00,113,368,1,190,14,20,Afternoon,Evening,"[360, 420)"
8357,YOW,Ottawa,YVR,Vancouver,3552.042539,Air Canada,223,2026-03-08,2026-05-20,13:55:00,Wednesday,5,2026-05-20,18:17:00,73,376,1,152,13,18,Afternoon,Evening,"[360, 420)"


In [4]:
df.describe()

,distance_km,month_departure,days_until_departure,trip_duration_minutes,number_of_stops,bookable_seats,base_price,departure_hour,arrival_hour
count,43157.000000,43157.000000,43157.000000,43157.000000,43157.000000,43157.000000,43157.000000,43157.000000,43157.000000
mean,2454.841528,6.177491,105.993280,290.655884,0.777232,7.858354,197.079222,13.023194,13.196005
std,1291.593992,2.103387,64.150674,116.924031,0.425416,1.642264,118.436908,5.523679,6.095380
min,363.703036,1.000000,0.000000,64.000000,0.000000,1.000000,22.000000,0.000000,0.000000
25%,686.261438,5.000000,55.000000,229.000000,1.000000,7.000000,130.000000,8.000000,9.000000
50%,3345.516314,6.000000,105.000000,325.000000,1.000000,9.000000,168.000000,13.000000,14.000000
75%,3552.042539,8.000000,154.000000,367.000000,1.000000,9.000000,224.000000,18.000000,19.000000
max,3552.042539,12.000000,357.000000,685.000000,2.000000,9.000000,1507.000000,23.000000,23.000000


In [5]:
df.head()

,origin,City,destination,City_destination,distance_km,Name_airline,aircraft,query_date,departure_date,departure_clock_time,day_of_week_departure,month_departure,arrival_date,arrival_clock_time,days_until_departure,trip_duration_minutes,number_of_stops,bookable_seats,base_price,departure_hour,arrival_hour,departure_time_period,arrival_time_period,duration_group
0,YOW,Ottawa,YYZ,Toronto,363.703036,WestJet,7M8,2026-03-08,2026-03-09,05:40:00,Monday,3,2026-03-09,07:01:00,1,81,0,7,216,5,7,Early Morning,Early Morning,"[60, 120)"
1,YOW,Ottawa,YYZ,Toronto,363.703036,Porter Airlines Canada Limited,295,2026-03-08,2026-03-09,06:45:00,Monday,3,2026-03-09,08:00:00,1,75,0,9,216,6,8,Early Morning,Morning,"[60, 120)"
2,YOW,Ottawa,YYZ,Toronto,363.703036,Porter Airlines Canada Limited,295,2026-03-08,2026-03-09,10:00:00,Monday,3,2026-03-09,11:15:00,1,75,0,9,216,10,11,Morning,Morning,"[60, 120)"
3,YOW,Ottawa,YYZ,Toronto,363.703036,Porter Airlines Canada Limited,295,2026-03-08,2026-03-09,15:05:00,Monday,3,2026-03-09,16:20:00,1,75,0,9,216,15,16,Afternoon,Afternoon,"[60, 120)"
4,YOW,Ottawa,YYZ,Toronto,363.703036,Porter Airlines Canada Limited,295,2026-03-08,2026-03-09,18:55:00,Monday,3,2026-03-09,20:10:00,1,75,0,9,216,18,20,Evening,Evening,"[60, 120)"


### III. Features to Create
- Weekend Flag: is_weekend = day_of_week_departure.isin(['Saturday', 'Sunday']).astype(int) (weekends often pricier).
- Season: season = month_departure.apply(lambda m: 'Winter' if m in [12,1,2] else 'Spring' if m in [3,4,5] else 'Summer' if m in [6,7,8] else 'Fall') (seasonal demand).
- Competition Level: Map from competition_df (e.g., 'Low', 'Medium', 'High' based on airline count per route).
- Days Bins: days_until_departure_bin = pd.cut(days_until_departure, bins=[0,10,40,120,max], labels=['Last Minute', 'Short', 'Optimal', 'Long']) (non-linear booking effects).
- Price per Minute: price_per_minute = total_price / trip_duration_minutes (route efficiency metric).
- Airline-Route Interaction: airline_route = Name_airline + '_' + (origin + '-' + destination) (captures airline-specific route pricing).
- Distance Bins: distance_bin = pd.cut(distance_km, bins=[0,500,1000,2000,max], labels=['Short', 'Medium', 'Long', 'Ultra']) (if non-linear distance effects).

In [ ]:
# creata a weekend feature
df['is_weekend_departure'] = df['day_of_week_departure'].isin(['Saturday', 'Sunday']).astype(int)

# seasonal feature
df['season'] = df['month_departure'].apply(lambda m: 'Winter' if m in [12,1,2] else 'Spring' if m in [3,4,5] else 'Summer' if m in [6,7,8] else 'Fall')

# days unitl departure group
# df['days_until_departure_group'] = pd.cut(df['days_until_departure'], 
# bins=[-1, 50, 100, 150, 200, 250, 300, 350, float('inf')], 
# labels=['0-50 days', '51-100 days', '101-150 days', '151-200 days', '200-250 days', '251-300 days', '301-350 days', '350+ days'])

# airline-route interaction feature
# df['airline_route'] = df['Name_airline'] + '_' + df['origin'] + '-' + df['destination']

# distanct bins feature
# df['distance_bin'] = pd.cut(df['distance_km'], 
# bins=[-1, 500, 1000, 1500, 2000, 2500, 3000, 3500, float('inf')], 
# labels=['0-500 km', '501-1000 km', '1001-1500 km', '1501-2000 km', '2001-2500 km', '2501-3000 km', '3001-3500 km', '3500+ km'])

### I. Features Selection
#### drop
- Redundant/High Correlation: City, City_destination (use origin/destination codes instead). month_departure (seasonal info better binned).
- Irrelevant/Noise: departure_clock_time, arrival_clock_time (already binned into periods). base_price if predicting total_price (avoid leakage).
- Low Impact: Name_airline outliers like First Air (already dropped). Any constant columns (e.g., if all flights are domestic).
- Post-EDA Drops: Temporary columns like route (recreate if needed).

#### keep
- Numerical: distance_km, days_until_departure, trip_duration_minutes, number_of_stops, bookable_seats (all correlate with price; bookable_seats indicates demand)
- Categorical: origin, destination, Name_airline, aircraft, day_of_week_departure, departure_time_period, arrival_time_period (capture route/airline/time effects).
- Target variable: base_price

In [18]:
df2 = df.loc[:,['base_price','origin','destination', 'distance_km',
       'Name_airline', 'day_of_week_departure', 'days_until_departure',
       'trip_duration_minutes', 'number_of_stops',
        'departure_hour', 'arrival_hour', 'departure_time_period',
       'arrival_time_period', 'is_weekend_departure',
       'season']]

df2.head(2)

,base_price,origin,destination,distance_km,Name_airline,day_of_week_departure,days_until_departure,trip_duration_minutes,number_of_stops,departure_hour,arrival_hour,departure_time_period,arrival_time_period,is_weekend_departure,season
0,216,YOW,YYZ,363.703036,WestJet,Monday,1,81,0,5,7,Early Morning,Early Morning,0,Spring
1,216,YOW,YYZ,363.703036,Porter Airlines Canada Limited,Monday,1,75,0,6,8,Early Morning,Morning,0,Spring


### II. Dummy Variables

In [19]:
df3 = pd.get_dummies(df2, columns = ['origin', 'destination', 'Name_airline', 'day_of_week_departure',
       'departure_time_period', 'arrival_time_period', 'season'
       ], drop_first=True).astype(int)

df3.head(2)

,base_price,distance_km,days_until_departure,trip_duration_minutes,number_of_stops,departure_hour,arrival_hour,is_weekend_departure,origin_YVR,origin_YYC,origin_YYZ,destination_YVR,destination_YYC,destination_YYZ,Name_airline_Air North Yukon's Airline,Name_airline_Air Transat,Name_airline_Central Mountain Air LTD,Name_airline_Pacific Coastal Airlines Limited,Name_airline_Porter Airlines Canada Limited,Name_airline_WestJet,day_of_week_departure_Monday,day_of_week_departure_Saturday,day_of_week_departure_Sunday,day_of_week_departure_Thursday,day_of_week_departure_Tuesday,day_of_week_departure_Wednesday,departure_time_period_Early Morning,departure_time_period_Evening,departure_time_period_Late Evening,departure_time_period_Morning,departure_time_period_Night,arrival_time_period_Early Morning,arrival_time_period_Evening,arrival_time_period_Late Evening,arrival_time_period_Morning,arrival_time_period_Night,season_Spring,season_Summer,season_Winter
0,216,363,1,81,0,5,7,0,0,0,0,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0
1,216,363,1,75,0,6,8,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0


In [20]:
# export csv
df3.to_csv('final_df.csv', index=False)

dropped:
- airline_route
- days_until_departure_group (redundant)
- distance_bin